# 00. Preparar bases — bronze Censo + CPF

Importa Censo + CPF, filtra UF/município, infere mãe no subset, CEP e logradouro (CNEFE).
Staging → join do nome fonético (`censo_pes_nome` / `cpf_cpf_nome`) → split
primeiro/meio/último → lista de ouro carimba `cpf_norm` no Censo.
Grava `censo_registros` / `cpf_registros` (sem empilhar).
`REBUILD` refaz o bronze; `REFILTER_GEO` reusa bronze e reconstrói registros.
Próximo: [`00b_limpar_dados.ipynb`](00b_limpar_dados.ipynb).


In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

%load_ext autoreload
%autoreload 2

REBUILD = False       # True = apaga bronze e reconstrói tudo
REFILTER_GEO = True   # True = reusa bronze, refaz filtro e reconstrói registros
REBUILD_REGISTROS = REBUILD or REFILTER_GEO

import config
from config import (
    CENSO_ESPECIE_ARQUIVO, CENSO_ENDERECO_ARQUIVO, CENSO_FACE_ARQUIVO,
    CENSO_LOGR_ARQUIVO,
    CENSO_PES_NOME_ARQUIVO, CENSO_PESSOAS_ARQUIVO, CPF_ARQUIVO, CPF_CPF_NOME_ARQUIVO,
    LISTA_OURO_ARQUIVO,
    CENSO_REGISTROS,
    CPF_REGISTROS,
    OUTPUT_DIR,
    TABELA_CENSO_REGISTROS,
    TABELA_CPF_REGISTROS,
    CENSO_COL_ID_DOMICILIO, CENSO_COL_ID_MORADOR, CENSO_COL_PRIMEIRO_NOME,
    CENSO_COL_SEXO, CENSO_COL_SOBRENOME, CPF_COL_CEP, CPF_COL_CPF,
    CPF_COL_DATA_NASC, CPF_COL_NOME, CPF_COL_NOME_MAE, CPF_COL_SEXO,
    CENSO_NOME_COL_ID, CENSO_NOME_COL_PHON, CENSO_NOME_COL_MAE_PHON,
    CPF_NOME_COL_CPF, CPF_NOME_COL_PHON, CPF_NOME_COL_MAE_PHON,
    benchmark_checkpoint, censo_cep_join_on, censo_dob_sql, censo_municipio_expr,
    cep_norm_sql, cpf_municipio_expr, cpf_norm_sql, cpf_uf_expr, censo_uf_expr,
    check_registros_vs_filtrado, export_parquet, geo_filter_clause, get_connection,
    idade_censo_sql, idade_cpf_sql, list_tables, materialize_censo_logr_lookup,
    materialize_censo_registros, materialize_cohort_cpf_por_censo, normalize_date_sql,
    print_paths, require_input, require_tables, stamp_censo_cpf_from_cohort,
)
from features import (
    NOME_MAE_COLUMNS,
    PESSOA_COLUMNS,
    clean_name_sql,
    name_feature_columns_sql,
    normalize_date_compacta_sql,
    normalize_sexo_sql,
    select_list_sql,
)
from inferir_pais import inferir_nome_mae_duckdb

# O filtro geográfico vem de config.py (escalar ou lista, um eixo por vez).
# Para sobrescrever só nesta sessão, descomente abaixo — reatribuir
# FILTRO_UF / FILTRO_MUNICIPIO aqui criaria apenas uma cópia local,
# sem efeito no filtro real. set_filtros também aponta OUTPUT_DIR para
# OUTPUT_DIR_BASE/<slug>/; use config.OUTPUT_DIR depois, não a cópia do import.
# config.set_filtros(municipio=2111300)
# config.set_filtros(uf=[21, 22])

print_paths()
for label, p in [
    ('CPF', CPF_ARQUIVO), ('CENSO_PESSOAS', CENSO_PESSOAS_ARQUIVO),
    ('CENSO_PES_NOME', CENSO_PES_NOME_ARQUIVO),
    ('CPF_CPF_NOME', CPF_CPF_NOME_ARQUIVO),
    ('CENSO_ESPECIE', CENSO_ESPECIE_ARQUIVO),
    ('CENSO_ENDERECO', CENSO_ENDERECO_ARQUIVO),
    ('CENSO_FACE', CENSO_FACE_ARQUIVO),
    ('CENSO_LOGR', CENSO_LOGR_ARQUIVO),
    ('LISTA_OURO', LISTA_OURO_ARQUIVO),
]:
    require_input(p, label=label)

con = get_connection()
print(
    'REBUILD:', REBUILD, '| REFILTER_GEO:', REFILTER_GEO,
    '| FILTRO_UF:', config.FILTRO_UF, '| FILTRO_MUNICIPIO:', config.FILTRO_MUNICIPIO,
)

if REBUILD:
    print('Rebuild completo: bronze + filtro + registros.')
elif REFILTER_GEO:
    print('REFILTER_GEO: reusa bronze, refaz filtro e reconstrói registros (00b lê registros, não *_filtrado).')
elif TABELA_CENSO_REGISTROS in list_tables(con) and TABELA_CPF_REGISTROS in list_tables(con):
    require_tables(con, [TABELA_CENSO_REGISTROS, TABELA_CPF_REGISTROS], notebook_origem='00')
    print('Tabelas finais já existem — defina REBUILD=True ou REFILTER_GEO=True para refazer.')
else:
    print('Prosseguir com pipeline completo nas células abaixo.')


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_especie.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_logr.parquet
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
LISTA_OURO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/scripts_luis/Pareamento Determinístico/V4 20260827/

## 1. Inspecionar bronze


In [2]:
# Só inspeciona o schema — não altera REBUILD do topo do notebook.
for label, path in [
    ('cpf', CPF_ARQUIVO),
    ('censo_pessoas', CENSO_PESSOAS_ARQUIVO),
    ('censo_pes_nome', CENSO_PES_NOME_ARQUIVO),
    ('cpf_cpf_nome', CPF_CPF_NOME_ARQUIVO),
    ('censo_especie', CENSO_ESPECIE_ARQUIVO),
    ('censo_endereco', CENSO_ENDERECO_ARQUIVO),
    ('censo_face', CENSO_FACE_ARQUIVO),
    ('censo_logr', CENSO_LOGR_ARQUIVO),
]:
    print(f'\n=== {label} ===')
    display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").df())



=== cpf ===


,column_name,column_type,null,key,default,extra
0,COD_CPF,VARCHAR,YES,None,None,None
1,NOM_PESSOA,VARCHAR,YES,None,None,None
2,NOM_SOCIAL,VARCHAR,YES,None,None,None
3,DAT_NASCIMENTO,VARCHAR,YES,None,None,None
4,COD_SEXO,VARCHAR,YES,None,None,None
5,NOM_MAE,VARCHAR,YES,None,None,None
6,COD_SITCAD,VARCHAR,YES,None,None,None
7,IND_RES_EXTERIOR,VARCHAR,YES,None,None,None
8,COD_PAIS_RES,VARCHAR,YES,None,None,None
9,NOM_PAIS_RES,VARCHAR,YES,None,None,None



=== censo_pessoas ===


,column_name,column_type,null,key,default,extra
0,B0000,"DECIMAL(17,0)",YES,None,None,None
1,B0001,VARCHAR,YES,None,None,None
2,B0002,VARCHAR,YES,None,None,None
3,B0003,VARCHAR,YES,None,None,None
4,B0004,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
133,CONC_URBANA,VARCHAR,YES,None,None,None
134,DOCA0105,VARCHAR,YES,None,None,None
135,PERE0104_NOVA,VARCHAR,YES,None,None,None
136,B0007_NOVO,FLOAT,YES,None,None,None



=== censo_pes_nome ===


,column_name,column_type,null,key,default,extra
0,ID_MORADOR,VARCHAR,YES,None,None,None
1,pes_nome,VARCHAR,YES,None,None,None
2,pes_nome_fonetico,VARCHAR,YES,None,None,None



=== cpf_cpf_nome ===


,column_name,column_type,null,key,default,extra
0,COD_CPF,VARCHAR,YES,None,None,None
1,cpf_nome,VARCHAR,YES,None,None,None
2,cpf_nome_fonetico,VARCHAR,YES,None,None,None


## 2. Importar bronze


In [3]:
IMPORTAR_BRONZE = True
if REBUILD and IMPORTAR_BRONZE:
    for tbl in [
        'cpf_bronze_raw', 'censo_pessoas_raw',
        'cpf_filtrado', 'censo_pessoas_filtrado',
        'censo_pais_inferidos', 'censo_logr_lookup', 'censo_morador_cep',
        'cpf_nome', 'censo_nome',
        'cpf_staging', 'cpf_registros',
        'censo_staging', 'censo_registros',
    ]:
        con.execute(f'DROP TABLE IF EXISTS {tbl}')

    con.execute(f"CREATE OR REPLACE TABLE cpf_bronze_raw AS SELECT * FROM read_parquet('{CPF_ARQUIVO}')")
    con.execute(f"CREATE OR REPLACE TABLE censo_pessoas_raw AS SELECT * FROM read_parquet('{CENSO_PESSOAS_ARQUIVO}')")

    for t in ['cpf_bronze_raw', 'censo_pessoas_raw']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 3. Filtrar por UF / município


In [4]:
# Diagnóstico: os dois lados precisam gerar código IBGE de 7 dígitos.
# Se COD_UFMUN vier com 6 dígitos, o lpad produz 0XXXXXX e o filtro erra.
if 'cpf_bronze_raw' in list_tables(con):
    display(con.execute(f'''
    SELECT length(regexp_replace(CAST("{config.CPF_COL_UF}" AS VARCHAR), '[^0-9]', '', 'g')) AS n_digitos,
           COUNT(*) AS n
    FROM cpf_bronze_raw
    GROUP BY 1 ORDER BY 2 DESC
    ''').df())
    display(con.execute(f'''
    SELECT {cpf_municipio_expr('c')} AS cod_municipio_cpf, COUNT(*) AS n
    FROM cpf_bronze_raw c GROUP BY 1 ORDER BY 2 DESC LIMIT 5
    ''').df())

if 'censo_pessoas_raw' in list_tables(con):
    display(con.execute(f'''
    SELECT {censo_municipio_expr('p')} AS cod_municipio_censo, COUNT(*) AS n
    FROM censo_pessoas_raw p GROUP BY 1 ORDER BY 2 DESC LIMIT 5
    ''').df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_digitos,n
0,7,257045117
1,<NA>,18849931


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cod_municipio_cpf,n
0,None,18849931
1,3550308,15585611
2,3304557,9548225
3,5300108,3712962
4,2927408,3476491


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cod_municipio_censo,n
0,3550308,11451999
1,3304557,6211223
2,5300108,2817381
3,2304400,2428708
4,2927408,2417678


In [5]:
if REBUILD_REGISTROS:
    CPF_UF = cpf_uf_expr('c')
    CENSO_UF = censo_uf_expr('p')
    CPF_MUN = cpf_municipio_expr('c')
    CENSO_MUN = censo_municipio_expr('p')
    cpf_where = geo_filter_clause(CPF_UF, CPF_MUN)
    censo_where = geo_filter_clause(CENSO_UF, CENSO_MUN)
    filtro_ativo = config.FILTRO_UF is not None or config.FILTRO_MUNICIPIO is not None
    if filtro_ativo and cpf_where == 'TRUE' and censo_where == 'TRUE':
        raise RuntimeError(
            'Filtro configurado mas nenhuma cláusula gerada — use config.set_filtros().'
        )
    print('FILTRO_UF:', config.FILTRO_UF, '| FILTRO_MUNICIPIO:', config.FILTRO_MUNICIPIO)
    print('CPF WHERE:', cpf_where)
    print('CENSO WHERE:', censo_where)

    if 'cpf_bronze_raw' in list_tables(con):
        n_antes = con.execute('SELECT COUNT(*) FROM cpf_bronze_raw').fetchone()[0]
    else:
        n_antes = None

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_filtrado AS
    SELECT c.* FROM cpf_bronze_raw c
    WHERE {cpf_where}
    ''')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_pessoas_filtrado AS
    SELECT p.* FROM censo_pessoas_raw p
    WHERE {censo_where}
    ''')

    for t in ['cpf_filtrado', 'censo_pessoas_filtrado']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')
    if n_antes is not None:
        n_depois = con.execute('SELECT COUNT(*) FROM cpf_filtrado').fetchone()[0]
        print(f'CPF: {n_antes:,} bronze → {n_depois:,} filtrado')
        if filtro_ativo and n_depois == n_antes:
            raise RuntimeError(
                f'Filtro ativo mas nada foi filtrado ({n_depois:,} = bronze). '
                'Confira o formato de COD_UFMUN na célula de diagnóstico.'
            )
        if filtro_ativo and n_depois == 0:
            raise RuntimeError(
                'Filtro ativo e resultado vazio — provável divergência de formato '
                'entre COD_UFMUN (CPF) e o prefixo do setor censitário.'
            )
elif not REBUILD:
    print('Filtro geográfico pulado — defina REFILTER_GEO=True ou REBUILD=True')


FILTRO_UF: ('21',) | FILTRO_MUNICIPIO: None
CPF WHERE: (substr(lpad(regexp_replace(CAST(c."COD_UFMUN" AS VARCHAR), '[^0-9]', '', 'g'), 7, '0'), 1, 2) = '21')
CENSO WHERE: (substr(CASE WHEN regexp_replace(CAST(p.B0000 AS VARCHAR), '[^0-9]', '', 'g') = '' THEN NULL ELSE lpad(regexp_replace(CAST(p.B0000 AS VARCHAR), '[^0-9]', '', 'g'), 15, '0') END, 1, 2) = '21')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] cpf_filtrado: 8447170
[checkpoint] censo_pessoas_filtrado: 6775805
CPF: 275,895,048 bronze → 8,447,170 filtrado


## 3b. Diagnóstico da idade

Independente de `REBUILD`: roda sobre as tabelas filtradas e mostra onde a
idade se perde de cada lado. No Censo a idade do linkage vem de **`PECP0401`**
(variável calculada, universo). `PECP0003`/`PECP0030` são do questionário
(amostra) e ficam só como contexto. No CPF a idade é derivada da data de
nascimento; a suspeita usual é o formato de `DAT_NASCIMENTO` não bater com o
que o `normalize_date_sql` espera.

In [6]:
tabelas = list_tables(con)
IDADE_CALC = config.CENSO_COL_IDADE_CALC
IDADE_QUEST = config.CENSO_COL_IDADE_ANOS_QUEST
IDADE_MESES = config.CENSO_COL_IDADE_MESES

if 'censo_pessoas_filtrado' in tabelas:
    print('=== CENSO: PECP0401 (idade do linkage) ===')
    display(con.execute(f'''
    DESCRIBE SELECT p."{IDADE_CALC}" AS {IDADE_CALC}
    FROM censo_pessoas_filtrado p LIMIT 0
    ''').df())

    IDADE_C = config.idade_censo_sql('p')
    display(con.execute(f'''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN p."{IDADE_CALC}" IS NULL THEN 1 ELSE 0 END) AS pecp0401_nulo,
        SUM(CASE WHEN {IDADE_C} IS NULL THEN 1 ELSE 0 END) AS idade_nula,
        ROUND(100.0 * SUM(CASE WHEN {IDADE_C} IS NULL THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS pct_idade_nula
    FROM censo_pessoas_filtrado p
    ''').df())

    print(f'--- {IDADE_CALC}: 12 valores crus mais frequentes ---')
    display(con.execute(f'''
    SELECT CAST(p."{IDADE_CALC}" AS VARCHAR) AS valor, COUNT(*) AS n
    FROM censo_pessoas_filtrado p GROUP BY 1 ORDER BY n DESC LIMIT 12
    ''').df())

    print('=== CENSO: questionário (PECP0003 / PECP0030), só contexto ===')
    display(con.execute(f'''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN p."{IDADE_QUEST}" IS NULL THEN 1 ELSE 0 END) AS pecp0003_nulo,
        SUM(CASE WHEN p."{IDADE_MESES}" IS NULL THEN 1 ELSE 0 END) AS pecp0030_nulo
    FROM censo_pessoas_filtrado p
    ''').df())

if 'cpf_filtrado' in tabelas:
    print('\n=== CPF: da data de nascimento até a idade ===')
    DT_BASE = normalize_date_sql(f'c."{CPF_COL_DATA_NASC}"')
    DT = normalize_date_compacta_sql(f'c."{CPF_COL_DATA_NASC}"')
    display(con.execute(f'''
    DESCRIBE SELECT c."{CPF_COL_DATA_NASC}" AS {CPF_COL_DATA_NASC}
    FROM cpf_filtrado c LIMIT 0
    ''').df())
    # 'recuperado_compacto' > 0 significa que a base guarda YYYYMMDD e que era
    # isso que zerava a idade do CPF.
    display(con.execute(f'''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN c."{CPF_COL_DATA_NASC}" IS NULL THEN 1 ELSE 0 END) AS raw_nulo,
        SUM(CASE WHEN {DT_BASE} = '' THEN 1 ELSE 0 END) AS nao_parseou_antes,
        SUM(CASE WHEN {DT} = '' THEN 1 ELSE 0 END) AS nao_parseou_agora,
        SUM(CASE WHEN {DT_BASE} = '' AND {DT} <> '' THEN 1 ELSE 0 END)
            AS recuperado_compacto,
        SUM(CASE WHEN {idade_cpf_sql(DT)} IS NULL THEN 1 ELSE 0 END) AS idade_nula,
        ROUND(100.0 * SUM(CASE WHEN {idade_cpf_sql(DT)} IS NULL THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS pct_idade_nula
    FROM cpf_filtrado c
    ''').df())
    print('--- DAT_NASCIMENTO: 12 valores crus mais frequentes ---')
    display(con.execute(f'''
    SELECT CAST(c."{CPF_COL_DATA_NASC}" AS VARCHAR) AS valor, COUNT(*) AS n
    FROM cpf_filtrado c GROUP BY 1 ORDER BY n DESC LIMIT 12
    ''').df())


=== CENSO: PECP0401 (idade do linkage) ===


,column_name,column_type,null,key,default,extra
0,PECP0401,"DECIMAL(11,0)",YES,None,None,None


,n,pecp0401_nulo,idade_nula,pct_idade_nula
0,6775805,0.0,5.0,0.0


--- PECP0401: 12 valores crus mais frequentes ---


,valor,n
0,15,125555
1,16,124874
2,17,124449
3,22,124211
4,18,122160
5,14,121517
6,13,118552
7,12,117042
8,19,115795
9,40,115543


...


=== CPF: da data de nascimento até a idade ===


,column_name,column_type,null,key,default,extra
0,DAT_NASCIMENTO,VARCHAR,YES,None,None,None


,n,raw_nulo,nao_parseou_antes,nao_parseou_agora,recuperado_compacto,idade_nula,pct_idade_nula
0,8447170,10.0,591.0,591.0,0.0,250631.0,2.97


--- DAT_NASCIMENTO: 12 valores crus mais frequentes ---


,valor,n
0,26/05/2002,550
1,20/03/2000,536
2,15/11/1986,535
3,09/09/2004,529
4,07/09/1985,519
5,26/08/2004,508
6,30/02/1970,505
7,22/03/2000,495
8,25/03/2000,493
9,07/09/1987,492


## 4. Inferir nome da mãe (Censo — **após filtro geográfico**)

Usa `censo_pessoas_filtrado` — não roda na base nacional inteira.


In [7]:
if REBUILD_REGISTROS:
    inferir_nome_mae_duckdb(con, source_table='censo_pessoas_filtrado')
    benchmark_checkpoint(con, 'censo_pais_inferidos', 'SELECT COUNT(*) FROM censo_pais_inferidos')
    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) AS com_mae,
        ROUND(100.0 * SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mae
    FROM censo_pais_inferidos
    ''').df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] censo_pais_inferidos: 6775805


## 5. CEP e logradouro Censo (espécie ⋈ endereço ⋈ face ⋈ LOGR — **após filtro geográfico**)

LEFT JOIN em `censo_pessoas_filtrado` por `B0000`, `NUM_QUADRA`, `NUM_FACE`, `B0006`, `COD_SEQ_ESPECIE`.

Lookup CNEFE: espécie → endereço → face (`cod_seglogr`) → LOGR (CEP, tipo, nome). Filtro UF/município entra no LOGR.


In [8]:
if REBUILD_REGISTROS:
    materialize_censo_logr_lookup(con)
    benchmark_checkpoint(con, 'censo_logr_lookup', 'SELECT COUNT(*) FROM censo_logr_lookup')

    join_on = censo_cep_join_on('p', 'k')
    con.execute(f'''
    CREATE OR REPLACE TABLE censo_morador_cep AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        COALESCE(k.cep, '') AS cep,
        COALESCE(k.tipo_logradouro, '') AS tipo_logradouro,
        COALESCE(k.logradouro, '') AS logradouro
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_logr_lookup k ON {join_on}
    ''')

    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) AS com_cep,
        ROUND(100.0 * SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cep,
        SUM(CASE WHEN logradouro <> '' THEN 1 ELSE 0 END) AS com_logradouro,
        ROUND(100.0 * SUM(CASE WHEN logradouro <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_logradouro
    FROM censo_morador_cep
    ''').df()


censo_logr_lookup: 663,835 linhas ESPECIE⋈LOGR, 663,835 faces distintas
[checkpoint] censo_logr_lookup: 663835


## 5b. Nomes fonéticos

Carrega `censo_pes_nome` e `cpf_cpf_nome`, confere chave única e guarda
só id + nome fonético. O join entra no staging.

In [9]:
if REBUILD_REGISTROS:
    cpf_nome_path = str(CPF_CPF_NOME_ARQUIVO).replace("'", "''")
    censo_nome_path = str(CENSO_PES_NOME_ARQUIVO).replace("'", "''")

    cpf_nome_cols = [
        r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{cpf_nome_path}') LIMIT 0"
        ).fetchall()
    ]
    print('cpf_cpf_nome colunas:', cpf_nome_cols)
    display(con.execute(
        f"SELECT * FROM read_parquet('{cpf_nome_path}') LIMIT 5"
    ).df())
    if CPF_NOME_COL_CPF not in cpf_nome_cols:
        raise KeyError(
            f'{CPF_NOME_COL_CPF} não está em cpf_cpf_nome. '
            f'Colunas: {cpf_nome_cols}. Ajuste CPF_NOME_COL_CPF em config.py.'
        )
    if CPF_NOME_COL_PHON not in cpf_nome_cols:
        raise KeyError(
            f'{CPF_NOME_COL_PHON} não está em cpf_cpf_nome. '
            f'Colunas: {cpf_nome_cols}. Ajuste CPF_NOME_COL_PHON em config.py.'
        )
    mae_cpf_sql = ''
    if CPF_NOME_COL_MAE_PHON:
        if CPF_NOME_COL_MAE_PHON not in cpf_nome_cols:
            raise KeyError(
                f'{CPF_NOME_COL_MAE_PHON} não está em cpf_cpf_nome. '
                f'Colunas: {cpf_nome_cols}.'
            )
        mae_cpf_sql = (
            f', NULLIF(TRIM(CAST("{CPF_NOME_COL_MAE_PHON}" AS VARCHAR)), \'\') '
            'AS nome_mae_phon'
        )
    CPF_N_NOME = cpf_norm_sql(f'"{CPF_NOME_COL_CPF}"')
    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_nome AS
    SELECT
        {CPF_N_NOME} AS cpf_norm,
        NULLIF(TRIM(CAST("{CPF_NOME_COL_PHON}" AS VARCHAR)), '') AS nome_completo_phon
        {mae_cpf_sql}
    FROM read_parquet('{cpf_nome_path}')
    WHERE {CPF_N_NOME} IS NOT NULL
    ''')
    n_cpf_nome, n_cpf_dist = con.execute(
        'SELECT COUNT(*), COUNT(DISTINCT cpf_norm) FROM cpf_nome'
    ).fetchone()
    print(f'cpf_nome: {n_cpf_nome:,} linhas | {n_cpf_dist:,} CPF distintos')
    if n_cpf_nome == 0:
        raise RuntimeError(
            f'{CPF_NOME_COL_CPF} não gerou cpf_norm válido. '
            'Sem dígito vira NULL, não 00000000000.'
        )
    if n_cpf_nome != n_cpf_dist:
        raise RuntimeError(
            f'cpf_cpf_nome tem CPF duplicado '
            f'({n_cpf_nome:,} linhas, {n_cpf_dist:,} distintos).'
        )

    censo_nome_cols = [
        r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{censo_nome_path}') LIMIT 0"
        ).fetchall()
    ]
    print('censo_pes_nome colunas:', censo_nome_cols)
    display(con.execute(
        f"SELECT * FROM read_parquet('{censo_nome_path}') LIMIT 5"
    ).df())
    if CENSO_NOME_COL_ID not in censo_nome_cols:
        raise KeyError(
            f'{CENSO_NOME_COL_ID} não está em censo_pes_nome. '
            f'Colunas: {censo_nome_cols}. Ajuste CENSO_NOME_COL_ID em config.py.'
        )
    if CENSO_NOME_COL_PHON not in censo_nome_cols:
        raise KeyError(
            f'{CENSO_NOME_COL_PHON} não está em censo_pes_nome. '
            f'Colunas: {censo_nome_cols}. Ajuste CENSO_NOME_COL_PHON em config.py.'
        )
    mae_censo_sql = ''
    if CENSO_NOME_COL_MAE_PHON:
        if CENSO_NOME_COL_MAE_PHON not in censo_nome_cols:
            raise KeyError(
                f'{CENSO_NOME_COL_MAE_PHON} não está em censo_pes_nome. '
                f'Colunas: {censo_nome_cols}.'
            )
        mae_censo_sql = (
            f', NULLIF(TRIM(CAST("{CENSO_NOME_COL_MAE_PHON}" AS VARCHAR)), \'\') '
            'AS nome_mae_phon'
        )
    ID_CENSO_NOME = f'CAST("{CENSO_NOME_COL_ID}" AS VARCHAR)'
    con.execute(f'''
    CREATE OR REPLACE TABLE censo_nome AS
    SELECT
        {ID_CENSO_NOME} AS person_id_censo,
        NULLIF(TRIM(CAST("{CENSO_NOME_COL_PHON}" AS VARCHAR)), '') AS nome_completo_phon
        {mae_censo_sql}
    FROM read_parquet('{censo_nome_path}')
    WHERE {ID_CENSO_NOME} IS NOT NULL AND TRIM({ID_CENSO_NOME}) <> ''
    ''')
    n_censo_nome, n_censo_dist = con.execute(
        'SELECT COUNT(*), COUNT(DISTINCT person_id_censo) FROM censo_nome'
    ).fetchone()
    print(
        f'censo_nome: {n_censo_nome:,} linhas | '
        f'{n_censo_dist:,} ID_MORADOR distintos'
    )
    if n_censo_nome != n_censo_dist:
        raise RuntimeError(
            f'censo_pes_nome tem ID_MORADOR duplicado '
            f'({n_censo_nome:,} linhas, {n_censo_dist:,} distintos).'
        )


cpf_cpf_nome colunas: ['COD_CPF', 'cpf_nome', 'cpf_nome_fonetico']


,COD_CPF,cpf_nome,cpf_nome_fonetico
0,00721286909,FERNANDA BATISTA DE ASSIS SOUZA,FERNANDA BATISTA ASIS SOUZA
1,00721287042,ANGELICA VARGAS,ANJELIKA VARGAS
2,00721287123,JOAO PAULO GOMES ALMEIDA,JOAU PAULU GOMIS AUMEIDA
3,00721287204,RENNAN RODRIGUES LIMA,RENAN RODRIGUIS LIMA
4,00721287395,LUIZA RIBEIRO DE SOUSA,LUIZA RIBEIRU SOUZA


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cpf_nome: 275,895,048 linhas | 275,895,048 CPF distintos
censo_pes_nome colunas: ['ID_MORADOR', 'pes_nome', 'pes_nome_fonetico']


,ID_MORADOR,pes_nome,pes_nome_fonetico
0,1100114050000580000160000410000001000000004,LUCAS BENTO OHNEZORG,LUKAS BENTU ONEZORGI
1,1100114050000580000160000411000001000000001,REGINA OHNEZORG,REJINA ONEZORGI
2,1100114050000580000160000411000001000000002,GEORGINA OHNEZORG,JEORJINA ONEZORGI
3,1100114050000580000160000411000001000000003,NATALIA OHNEZORG DE SOUZA,NATALIA ONEZORGI SOUZA
4,1100114050000580000160000411000001000000004,EDEIUSON ROCHA DE SOUZA,EDEIUZON ROXA SOUZA


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

censo_nome: 195,101,203 linhas | 195,101,203 ID_MORADOR distintos


## 6. CPF — staging

Colunas brutas (nome, data, CEP, óbito) + `nome_completo_phon` do join com
`cpf_cpf_nome`. Split de primeiro/último fica na seção de nomes.


In [10]:
if REBUILD_REGISTROS:
    CPF_N = cpf_norm_sql(f'c."{CPF_COL_CPF}"')
    DT_NASC = normalize_date_compacta_sql(f'c."{CPF_COL_DATA_NASC}"')
    NOME_MAE = f'c."{CPF_COL_NOME_MAE}"'
    SEXO = f'c."{CPF_COL_SEXO}"'
    CEP = cep_norm_sql(f'c."{CPF_COL_CEP}"')
    UF_COL = cpf_uf_expr('c')
    MUN_COL = cpf_municipio_expr('c')

    # ANO_OBITO existe no bronze do CPF. Falha aqui é melhor que seguir com a
    # coluna nula, que desligaria o filtro de óbito do NB00b sem avisar.
    cpf_cols = set(con.execute('SELECT * FROM cpf_filtrado LIMIT 0').df().columns)
    if config.CPF_COL_ANO_OBITO not in cpf_cols:
        candidatas = sorted(c for c in cpf_cols if 'OBITO' in c.upper())
        raise KeyError(
            f'{config.CPF_COL_ANO_OBITO} não existe no CPF bronze. '
            f'Candidatas: {candidatas or "nenhuma"}. '
            'Ajuste CPF_COL_ANO_OBITO em config.py.'
        )
    ANO_OBITO = f'TRY_CAST(c."{config.CPF_COL_ANO_OBITO}" AS INTEGER)'
    PHON_PESSOA = f'NULLIF(TRIM(CAST(n.nome_completo_phon AS VARCHAR)), \'\')'
    MAE_PHON_SEL = (
        ", NULLIF(TRIM(CAST(n.nome_mae_phon AS VARCHAR)), '') AS nome_mae_phon"
        if CPF_NOME_COL_MAE_PHON else ""
    )

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_staging AS
    SELECT
        {CPF_N} AS cpf_norm,
        TRIM(CAST(c."{CPF_COL_NOME}" AS VARCHAR)) AS nome_completo_raw,
        {DT_NASC} AS data_nascimento,
        {idade_cpf_sql(DT_NASC)} AS idade,
        CAST({NOME_MAE} AS VARCHAR) AS nome_mae_raw,
        CAST({SEXO} AS VARCHAR) AS sexo_raw,
        {CEP} AS cep,
        {UF_COL} AS uf,
        {MUN_COL} AS cod_municipio,
        {ANO_OBITO} AS ano_obito,
        {PHON_PESSOA} AS nome_completo_phon
        {MAE_PHON_SEL}
    FROM cpf_filtrado c
    LEFT JOIN cpf_nome n ON {CPF_N} = n.cpf_norm
    WHERE {CPF_N} IS NOT NULL
    ''')
    benchmark_checkpoint(con, 'cpf_staging', 'SELECT COUNT(*) FROM cpf_staging')
    n_cpf, n_phon = con.execute('''
    SELECT COUNT(*), COUNT(nome_completo_phon) FROM cpf_staging
    ''').fetchone()
    print(
        f'cpf_staging: {n_cpf:,} linhas | '
        f'com nome_completo_phon: {n_phon:,} '
        f'({100.0 * n_phon / n_cpf if n_cpf else 0:.1f}%)'
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] cpf_staging: 8447170
cpf_staging: 8,447,170 linhas | com nome_completo_phon: 8,447,170 (100.0%)


## 7. Censo — staging

Pessoa + CEP + tipo/nome do logradouro + mãe inferida + `nome_completo_phon` do join com `censo_pes_nome`.
Sem lista de ouro ainda: `cpf_norm` entra depois.


In [11]:
if REBUILD_REGISTROS:
    DT_PESSOA = censo_dob_sql()
    DT_NASC_C = normalize_date_sql(DT_PESSOA)
    NOME_COMPLETO = (
        f"TRIM(COALESCE(CAST(p.{CENSO_COL_PRIMEIRO_NOME} AS VARCHAR), '')"
        f" || ' ' || COALESCE(CAST(p.{CENSO_COL_SOBRENOME} AS VARCHAR), ''))"
    )
    UF_C = censo_uf_expr('p')
    MUN_C = censo_municipio_expr('p')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_staging AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        CAST(p.{CENSO_COL_ID_DOMICILIO} AS VARCHAR) AS id_domicilio,
        {NOME_COMPLETO} AS nome_completo_raw,
        {DT_NASC_C} AS data_nascimento,
        {idade_censo_sql('p')} AS idade,
        CAST(p.{CENSO_COL_SEXO} AS VARCHAR) AS sexo_raw,
        {UF_C} AS uf,
        {MUN_C} AS cod_municipio,
        COALESCE(e.cep, '') AS cep,
        COALESCE(e.tipo_logradouro, '') AS tipo_logradouro,
        COALESCE(e.logradouro, '') AS logradouro,
        m.nome_mae AS nome_mae_inferido,
        NULLIF(TRIM(CAST(n.nome_completo_phon AS VARCHAR)), '') AS nome_completo_phon
        {", NULLIF(TRIM(CAST(n.nome_mae_phon AS VARCHAR)), '') AS nome_mae_phon" if CENSO_NOME_COL_MAE_PHON else ""}
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_morador_cep e
        ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = e.person_id_censo
    LEFT JOIN censo_pais_inferidos m
        ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = m.person_id_censo
    LEFT JOIN censo_nome n
        ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = n.person_id_censo
    ''')
    benchmark_checkpoint(con, 'censo_staging', 'SELECT COUNT(*) FROM censo_staging')
    n_censo, n_phon = con.execute('''
    SELECT COUNT(*), COUNT(nome_completo_phon) FROM censo_staging
    ''').fetchone()
    print(
        f'censo_staging: {n_censo:,} linhas | '
        f'com nome_completo_phon: {n_phon:,} '
        f'({100.0 * n_phon / n_censo if n_censo else 0:.1f}%)'
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] censo_staging: 6775805
censo_staging: 6,775,805 linhas | com nome_completo_phon: 6,589,804 (97.3%)


## 8. Nomes (split)

`nome_completo` permanece o limpo. `*_phon` já veio do join; aqui só parte
primeiro / meio / último (e o composto primeiro+último) no limpo e no fonético.


In [12]:
if REBUILD_REGISTROS:
    SEXO_N = normalize_sexo_sql('sexo_raw')
    cpf_cols = {r[0] for r in con.execute('DESCRIBE cpf_staging').fetchall()}
    mae_phon_cpf = (
        'nome_mae_phon' if 'nome_mae_phon' in cpf_cols else 'nome_mae_norm'
    )
    PESSOA = name_feature_columns_sql(
        'nome_completo_norm', col_map=PESSOA_COLUMNS, phon_col='nome_completo_phon'
    )
    MAE = {
        alias: f"NULLIF({expr}, '')"
        for alias, expr in name_feature_columns_sql(
            'nome_mae_norm', col_map=NOME_MAE_COLUMNS, phon_col=mae_phon_cpf
        ).items()
    }

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_registros AS
    WITH norm AS (
        SELECT *,
            {clean_name_sql('nome_completo_raw')} AS nome_completo_norm,
            {clean_name_sql('nome_mae_raw')} AS nome_mae_norm
        FROM cpf_staging
    )
    SELECT
        'cpf_' || cpf_norm AS unique_id, 'cpf' AS origem, cpf_norm,
        {select_list_sql(PESSOA)},
        data_nascimento,
        {select_list_sql(MAE)},
        {SEXO_N} AS sexo,
        idade,
        cep, CAST(uf AS VARCHAR) AS uf,
        CAST(cod_municipio AS VARCHAR) AS cod_municipio,
        ano_obito,
        CAST(NULL AS VARCHAR) AS person_id_censo,
        CAST(NULL AS VARCHAR) AS id_domicilio
    FROM norm
    ''')
    benchmark_checkpoint(con, 'cpf_registros', 'SELECT COUNT(*) FROM cpf_registros')
    materialize_censo_registros(con)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] cpf_registros: 8447170


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[checkpoint] censo_registros: 6775805


## 9. Lista de ouro (carimbo de CPF)

Carimba `cpf_norm` no Censo a partir de `LISTA_OURO_ARQUIVO`
(`PERSON_ID_CENSO` + `CPF_NORM`). Fora da lista fica NULL. O Splink só vê o carimbo depois dos nomes
prontos.


In [13]:
if REBUILD_REGISTROS:
    cohort_cpf = materialize_cohort_cpf_por_censo(
        con, cohort_parquet=LISTA_OURO_ARQUIVO
    )
    print(
        'Censos com CPF na lista de ouro:',
        f"{cohort_cpf['n_censo_com_cpf_coorte']:,}",
        '| ambíguos (MIN):',
        f"{cohort_cpf['n_censo_cpf_ambiguo']:,}",
    )
    n_censo_cpf = stamp_censo_cpf_from_cohort(
        con, table=TABELA_CENSO_REGISTROS
    )
    print(f'censo_registros com cpf_norm: {n_censo_cpf:,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Censos com CPF na lista de ouro: 19,683,974 | ambíguos (MIN): 0
censo_registros com cpf_norm: 510,774


## 10. Conferir as duas bases

Sem empilhar: o linkage (`link_only`) consome `censo_registros` e `cpf_registros`
separadas. `cpf_norm` no Censo veio da lista de ouro (NULL fora dela).


In [14]:
if REBUILD_REGISTROS:
    n_c = con.execute('SELECT COUNT(*) FROM censo_registros').fetchone()[0]
    n_p = con.execute('SELECT COUNT(*) FROM cpf_registros').fetchone()[0]
    n_c_cpf = con.execute(
        'SELECT COUNT(*) FROM censo_registros WHERE cpf_norm IS NOT NULL'
    ).fetchone()[0]
    print(f'censo_registros: {n_c:,} ({n_c_cpf:,} com CPF da lista de ouro)')
    print(f'cpf_registros: {n_p:,}')


censo_registros: 6,775,805 (510,774 com CPF da lista de ouro)
cpf_registros: 8,447,170


## 11. Export


In [15]:
if REBUILD_REGISTROS:
    p_c = export_parquet(con, 'censo_registros', path=CENSO_REGISTROS)
    p_p = export_parquet(con, 'cpf_registros', path=CPF_REGISTROS)
    print('Exportado:', p_c)
    print('Exportado:', p_p)
check_registros_vs_filtrado(con)
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exportado: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/censo_registros.parquet
Exportado: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/cpf_registros.parquet
censo_pessoas_filtrado: 6,775,805 | censo_registros: 6,775,805
cpf_filtrado: 8,447,170 | cpf_registros: 8,447,170
